### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [18]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


/tmp/ipykernel_5564/3805701412.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [25]:
def process_all_pdf(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    #print(pdf_files)

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata 
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

#process all pdfs in the data directory
all_pdf_documents = process_all_pdf("../data")



Found 3 PDF files to process

Processing: Cal Poly Slo Decision Letter.pdf
 Loaded 1 pages

Processing: Alice B. Hansen Scholarship Form.pdf
 Loaded 1 pages

Processing: written assignment 3.pdf
 Loaded 5 pages

 Total documents loaded: 7


In [26]:
all_pdf_documents

[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

In [27]:
###Text splitting (get into chunks)

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better Rag Performances"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n", "\n", " ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [28]:
chunks = split_documents(all_pdf_documents)
chunks

Split 7 documents into 9 chunks

Example chunk:
Content: 1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  
 
 
April 1, 2025 
 
Victor Xie 
1516 Buena Vista Ave Apt A 
Alameda, CA 94501-1218
Congratul...
Metadata: {'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

### embedding and vectorStoreDB

In [23]:
import numpy as np
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model successfully loaded. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model Not Loaded")
        print(f"Generating  embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

##initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1696.82it/s]


Model successfully loaded. Embedding dimension: 384


### VectorStore

In [32]:
class VectorStore:
    """Mangaes document embeddings in a ChromaDB vector store """
    def __init__(self, collection_name = "pdf_documents", persist_directory = "../data/vector_store"):
        """Initialize the vector store"""

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok= True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for Rag"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vectore store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add doucuments and their embeddings to the vector store"""

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print (f"Adding {len(documents)} documents to vector store...")
    
        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_text.append(doc.page_content)
            
            #Embedding
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,                
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vectore store: {e}")
            raise


vectorstore = VectorStore()
vectorstore
            

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 9


In [29]:
chunks

[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

In [31]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate the embeddings 
embeddings = embedding_manager.generate_embeddings(texts)
embeddings

#store in the vector datavase
vectorstore.add_documents(chunks,embeddings)

Generating  embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]

Generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Sucessfully added 9 documents to vector store
Total documents in collection: 9


### Retriever PipeLine From VectorStore